# Vígil.ia — Comparativo YOLO11s-det vs RT-DETR-l (até o vídeo)

**Objetivo:** decidir o modelo do modo LOCAL (notebook i3-1315U, sem servidor).
O RT-DETR-l venceu em qualidade mas pesa 103 GFLOPs (~1 fps em CPU). O YOLO11s
tem ~21 GFLOPs (5× mais leve). Treinamos o YOLO11s **no MESMO dataset v3**
(balanceado + blur + cenas multi-grão) e comparamos de forma justa:

1. **mAP** no mesmo split de validação (por classe também)
2. **Velocidade** GPU e CPU (ONNX)
3. **Vídeo**: o MESMO `teste_soja.mp4`, os dois com NMS + veredito travado → 2 vídeos anotados no Drive

O experimento decide — igual fizemos EfficientNet vs YOLO na classificação.

Pré-requisitos no Drive: fotos reais, `soja_rtdetr_ft_v3.pt`, `teste_soja.mp4`.

## 1. Setup

In [ ]:
!pip -q install "ultralytics==8.4.80" onnxruntime

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'Sem GPU!'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Caminhos

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

REAL_SRCS = [
    '/content/drive/MyDrive/Soja total/Soja total/Lotes',
    '/content/drive/MyDrive/Soja pra completar',
]

# RT-DETR fine-tunado v3 (o campeão atual)
RTDETR_PT = '/content/runs/detect/runs_rtdetr/ft_v3/weights/best.pt'
if not os.path.exists(RTDETR_PT):
    RTDETR_PT = '/content/drive/MyDrive/soja_rtdetr_ft_v3.pt'
assert os.path.exists(RTDETR_PT), 'soja_rtdetr_ft_v3.pt não encontrado no Drive!'
print('RT-DETR:', RTDETR_PT)

VIDEO_TESTE = '/content/drive/MyDrive/teste_soja.mp4'

## 3. Dataset v3 (reconstrói se a sessão for nova)
Mesmas funções do notebook de melhoria — balanceamento + blur + cenas multi-grão.

In [ ]:
import glob, hashlib, unicodedata, cv2, yaml
import numpy as np

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']
ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'], 4: ['spotted', 'manchad']}
IGNORE = ['part of the original']
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
RNG = np.random.default_rng(42)

def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()

def class_of(folder):
    n = norm(folder)
    if any(norm(k) in n for k in IGNORE):
        return None
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None

def collect_real(srcs, val_frac=0.15):
    items = []
    for src in srcs:
        for root, _, files in os.walk(src):
            cls = None
            for part in reversed(root.split(os.sep)):
                c = class_of(part)
                if c is not None:
                    cls = c; break
            if cls is None:
                continue
            for fn in files:
                if fn.lower().endswith(IMG_EXT):
                    p = os.path.join(root, fn)
                    h = int(hashlib.md5(p.encode()).hexdigest(), 16)
                    items.append((p, cls, 'val' if (h % 100) < val_frac * 100 else 'train'))
    from collections import Counter
    print('coletado:', dict(Counter(sp for _, _, sp in items)))
    return items

def sat_box(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def otsu_box(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < 0.005 * h * w or area > 0.995 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def letterbox640(img, size=640):
    h, w = img.shape[:2]
    s = size / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))))
    h, w = img.shape[:2]
    top, left = (size - h) // 2, (size - w) // 2
    img = cv2.copyMakeBorder(img, top, size - h - top, left, size - w - left,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return img, s, left, top

def motion_blur(img, rng=RNG):
    k = int(rng.choice([7, 9, 11, 13, 15]))
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(0, 180)), 1)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= max(kernel.sum(), 1e-6)
    return cv2.filter2D(img, -1, kernel)

def extract_cutout(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    mask = np.zeros((h, w), np.uint8)
    cv2.drawContours(mask, [c], -1, 255, -1)
    x, y, bw, bh = cv2.boundingRect(c)
    return img[y:y + bh, x:x + bw], mask[y:y + bh, x:x + bw]

def make_scene(cutouts, rng=RNG, size=640):
    bg = int(rng.integers(20, 130))
    canvas = np.clip(np.full((size, size, 3), bg, np.int16)
                     + rng.normal(0, 6, (size, size, 3)), 0, 255).astype(np.uint8)
    occ = np.zeros((size, size), np.uint8)
    boxes = []
    for _ in range(int(rng.integers(6, 26))):
        cls, crop, mask = cutouts[int(rng.integers(len(cutouts)))]
        s = int(rng.integers(60, 150)) / max(crop.shape[:2])
        crop2 = cv2.resize(crop, None, fx=s, fy=s)
        mask2 = cv2.resize(mask, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
        h2, w2 = crop2.shape[:2]
        diag = int(np.ceil(np.hypot(h2, w2))) + 2
        M = cv2.getRotationMatrix2D((w2 / 2, h2 / 2), float(rng.uniform(0, 360)), 1)
        M[0, 2] += (diag - w2) / 2
        M[1, 2] += (diag - h2) / 2
        crop3 = cv2.warpAffine(crop2, M, (diag, diag))
        mask3 = cv2.warpAffine(mask2, M, (diag, diag), flags=cv2.INTER_NEAREST)
        ys, xs = np.where(mask3 > 0)
        if not len(xs):
            continue
        crop3 = crop3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        mask3 = mask3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        gh, gw = mask3.shape
        if gh >= size - 2 or gw >= size - 2:
            continue
        placed = False
        for _try in range(20):
            px = int(rng.integers(0, size - gw))
            py = int(rng.integers(0, size - gh))
            inter = (occ[py:py + gh, px:px + gw] > 0) & (mask3 > 0)
            if inter.sum() <= 0.15 * (mask3 > 0).sum():
                placed = True
                break
        if not placed:
            continue
        alpha = (cv2.GaussianBlur(mask3, (5, 5), 0).astype(np.float32) / 255)[..., None]
        reg = canvas[py:py + gh, px:px + gw]
        canvas[py:py + gh, px:px + gw] = (alpha * crop3 + (1 - alpha) * reg).astype(np.uint8)
        occ[py:py + gh, px:px + gw][mask3 > 0] = 255
        boxes.append((cls, (px + gw / 2) / size, (py + gh / 2) / size, gw / size, gh / size))
    return canvas, boxes

def balance_train(items):
    from collections import defaultdict
    train = [it for it in items if it[2] == 'train']
    rest = [it for it in items if it[2] != 'train']
    by = defaultdict(list)
    for it in train:
        by[it[1]].append(it)
    mx = max(len(v) for v in by.values())
    out = []
    for c, v in by.items():
        out += v + [v[int(i)] for i in RNG.integers(0, len(v), mx - len(v))]
    print('balanceado (train):', {NAMES[c]: sum(1 for it in out if it[1] == c) for c in sorted(by)})
    return out + rest

def build_v3(items, out_dir, n_synth=600, blur_frac=0.4):
    assert items, 'Nenhuma imagem coletada! Confira REAL_SRCS.'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    items = balance_train(items)
    cutouts = []
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 200 == 0:
            print(f'  fotos {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        h0, w0 = img.shape[:2]
        box = sat_box(img) or otsu_box(img)
        if box is None:
            skipped += 1; continue
        if sp == 'train':
            cut = extract_cutout(img)
            if cut is not None:
                cutouts.append((cls, cut[0], cut[1]))
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        line = f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}'
        stem = f'{sp}_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(line)
        kept += 1
        if sp == 'train':
            cv2.imwrite(f'{out_dir}/images/train/{stem}b.jpg', motion_blur(lb),
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            open(f'{out_dir}/labels/train/{stem}b.txt', 'w').write(line)
            kept += 1
    print(f'fotos reais: kept={kept} skipped={skipped} | recortes: {len(cutouts)}')
    assert cutouts, 'Nenhum recorte extraído!'
    synth = 0
    for j in range(n_synth):
        if j % 100 == 0:
            print(f'  cenas {j}/{n_synth}…', flush=True)
        canvas, boxes = make_scene(cutouts)
        if not boxes:
            continue
        if RNG.random() < blur_frac:
            canvas = motion_blur(canvas)
        stem = f'synth_{j:05d}'
        cv2.imwrite(f'{out_dir}/images/train/{stem}.jpg', canvas, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/train/{stem}.txt', 'w').write(
            '\n'.join(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}' for c, cx, cy, w, h in boxes))
        synth += 1
    print(f'cenas sintéticas: {synth}')
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    return f'{out_dir}/data.yaml'

V3_YAML = '/content/soja_det_v3/data.yaml'
if not os.path.exists(V3_YAML):
    V3_YAML = build_v3(collect_real(REAL_SRCS), '/content/soja_det_v3')
print('dataset:', V3_YAML)

## 4. Treino — YOLO11s-detect no MESMO dataset v3
Parte do `yolo11s.pt` (COCO). Aqui warmup e AMP padrão funcionam bem
(os problemas de colapso eram específicos do RT-DETR).

In [ ]:
from ultralytics import YOLO

yolo = YOLO('yolo11s.pt')
yolo.train(
    data=V3_YAML, epochs=60, imgsz=640, batch=32, device=0, seed=42,
    optimizer='AdamW', lr0=0.001, patience=20, close_mosaic=8,
    cache=False, workers=8,
    mosaic=1.0, hsv_v=0.5, degrees=15, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    project='runs_cmp', name='yolo11s_v3', exist_ok=True,
)

!cp /content/runs/detect/runs_cmp/yolo11s_v3/weights/best.pt /content/drive/MyDrive/soja_yolo11s_det_v3.pt
print('backup ok: soja_yolo11s_det_v3.pt')

## 5. Validação lado a lado (mesmo split de val)

In [ ]:
from ultralytics import YOLO, RTDETR

YOLO_PT = '/content/runs/detect/runs_cmp/yolo11s_v3/weights/best.pt'
models = {
    'yolo11s': YOLO(YOLO_PT),
    'rtdetr-l': RTDETR(RTDETR_PT),
}

rows = {}
for tag, m in models.items():
    r = m.val(data=V3_YAML, split='val', imgsz=640, device=0, verbose=False)
    rows[tag] = r
    print(f"\n=== {tag} ===  mAP50={r.box.map50:.3f}  mAP50-95={r.box.map:.3f}"
          f"  ({sum(r.speed.values()):.1f} ms/img GPU)")
    for i, ap in zip(r.box.ap_class_index, r.box.maps[r.box.ap_class_index]):
        print(f'  {NAMES[int(i)]:14s} {ap:.3f}')

## 6. Velocidade em CPU (ONNX) — proxy do notebook i3
A CPU do Colab não é o i3-1315U, mas a RAZÃO entre os dois modelos se mantém.

In [ ]:
import time
import numpy as np
import onnxruntime as ort

paths = {}
paths['yolo11s'] = YOLO(YOLO_PT).export(format='onnx', imgsz=640, opset=12, simplify=True)
paths['rtdetr-l'] = RTDETR(RTDETR_PT).export(format='onnx', imgsz=640, opset=16, simplify=True)

x = np.random.rand(1, 3, 640, 640).astype(np.float32)
cpu_ms = {}
for tag, p in paths.items():
    sess = ort.InferenceSession(p, providers=['CPUExecutionProvider'])
    inp = sess.get_inputs()[0].name
    for _ in range(3):
        sess.run(None, {inp: x})  # warmup
    t0 = time.perf_counter()
    N = 15
    for _ in range(N):
        sess.run(None, {inp: x})
    ms = (time.perf_counter() - t0) / N * 1000
    cpu_ms[tag] = ms
    print(f'{tag:10s} CPU: {ms:7.1f} ms/frame  (~{1000/ms:.1f} fps nesta CPU)')

ratio = cpu_ms['rtdetr-l'] / cpu_ms['yolo11s']
print(f'\nYOLO11s é {ratio:.1f}x mais rápido em CPU.')

## 7. Patch NMS do RT-DETR (o YOLO já tem NMS nativo)

In [ ]:
import torchvision
from ultralytics.models.rtdetr.predict import RTDETRPredictor

if not hasattr(RTDETRPredictor, '_pp_orig'):
    RTDETRPredictor._pp_orig = RTDETRPredictor.postprocess

def _pp_nms(self, preds, img, orig_imgs):
    results = RTDETRPredictor._pp_orig(self, preds, img, orig_imgs)
    for r in results:
        if len(r.boxes) > 1:
            keep = torchvision.ops.nms(r.boxes.xyxy, r.boxes.conf, iou_threshold=0.6)
            r.update(boxes=r.boxes.data[keep])
    return results

RTDETRPredictor.postprocess = _pp_nms
print('patch NMS (RT-DETR) aplicado ✅')

## 8. Vídeo A/B — os dois com veredito travado, no MESMO vídeo
Gera `video_yolo11s_travado.mp4` e `video_rtdetr_travado.mp4` no Drive.
O YOLO usa `agnostic_nms=True` (mata caixa dupla de classes diferentes).

In [ ]:
from collections import defaultdict, Counter
import cv2

LOCK_MIN_FRAMES = 8
LOCK_RATIO = 0.60
COLORS = {
    'intact': (80, 200, 80), 'immature': (60, 200, 200),
    'broken': (200, 100, 160), 'skin-damaged': (60, 160, 255),
    'spotted': (80, 80, 230),
}

def locked_video(model, out_path, agnostic=False, conf=0.35):
    votes = defaultdict(Counter)
    frames_seen = Counter()
    locked = {}
    cap = cv2.VideoCapture(VIDEO_TESTE)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    cap.release()
    writer = None
    kw = dict(conf=conf, tracker='bytetrack.yaml', stream=True, persist=True, verbose=False)
    if agnostic:
        kw['agnostic_nms'] = True
    t0 = cv2.getTickCount()
    nframes = 0
    for r in model.track(source=VIDEO_TESTE, **kw):
        frame = r.orig_img.copy()
        nframes += 1
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
        if r.boxes.id is not None:
            for xyxy, tid, c, cf in zip(r.boxes.xyxy.cpu().numpy().astype(int),
                                        r.boxes.id.int().tolist(),
                                        r.boxes.cls.int().tolist(),
                                        r.boxes.conf.tolist()):
                x1, y1, x2, y2 = xyxy
                if tid not in locked:
                    votes[tid][NAMES[c]] += cf
                    frames_seen[tid] += 1
                    top, n = votes[tid].most_common(1)[0]
                    if frames_seen[tid] >= LOCK_MIN_FRAMES and n >= LOCK_RATIO * sum(votes[tid].values()):
                        locked[tid] = top
                if tid in locked:
                    cls = locked[tid]
                    color = COLORS[cls]
                    label = f'#{tid} {cls}'
                else:
                    color = (160, 160, 160)
                    label = f'#{tid} analisando…'
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, max(18, y1 - 6)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
        writer.write(frame)
    writer.release()
    secs = (cv2.getTickCount() - t0) / cv2.getTickFrequency()
    for tid, cnt in votes.items():
        locked.setdefault(tid, cnt.most_common(1)[0][0])
    intact = sum(1 for c in locked.values() if c == 'intact')
    pct = intact / len(locked) if locked else 0
    print(f'  grãos: {len(locked)} | intactos: {intact} ({pct*100:.0f}%)'
          f' -> {"APROVADO" if pct >= 0.9 else "REPROVADO"} | {nframes/secs:.1f} fps na GPU')
    return locked

print('YOLO11s:')
locked_yolo = locked_video(models['yolo11s'], '/content/video_yolo11s_travado.mp4', agnostic=True)
print('RT-DETR:')
locked_rtd = locked_video(models['rtdetr-l'], '/content/video_rtdetr_travado.mp4')

!cp /content/video_yolo11s_travado.mp4 /content/video_rtdetr_travado.mp4 /content/drive/MyDrive/
print('\n2 vídeos no Drive: video_yolo11s_travado.mp4 e video_rtdetr_travado.mp4')

## 9. Placar final
Junta tudo numa tabela. O juiz de qualidade é o VÍDEO (assista os dois);
os números dão o contexto de velocidade/precisão.

In [ ]:
print(f"{'':12s} {'mAP50':>7s} {'mAP50-95':>9s} {'GPU ms':>7s} {'CPU ms':>8s} {'CPU fps':>8s}")
for tag in ('yolo11s', 'rtdetr-l'):
    r = rows[tag]
    gpu = sum(r.speed.values())
    print(f'{tag:12s} {r.box.map50:7.3f} {r.box.map:9.3f} {gpu:7.1f}'
          f' {cpu_ms[tag]:8.1f} {1000/cpu_ms[tag]:8.1f}')

print('\nvereditos por grão (voto travado):')
ids = sorted(set(locked_yolo) | set(locked_rtd))
diff = 0
for tid in ids:
    a = locked_yolo.get(tid, '—')
    b = locked_rtd.get(tid, '—')
    mark = '' if a == b else '  <-- divergem'
    if a != b:
        diff += 1
    print(f'  #{tid:3d}: yolo={a:13s} rtdetr={b:13s}{mark}')
print(f'\n(IDs de rastreio podem não coincidir entre os modelos — compare pelos VÍDEOS.)')

## 10. Critério de decisão

- **YOLO11s empata (ou quase) em mAP e no vídeo** → ele leva o modo local: exportamos
  p/ OpenVINO INT8 e o i3 roda a ~10–15 fps.
- **RT-DETR visivelmente melhor no vídeo** → decisão de produto: qualidade (RT-DETR no
  servidor/GPU) vs praticidade (YOLO11s local). Dá pra ter os dois: YOLO local ao vivo
  + RT-DETR como "segunda opinião" em foto.

No i3-1315U, a expectativa de fps local é ~1 fps (RT-DETR) vs ~5–8 fps ONNX /
~10–15 fps OpenVINO INT8 (YOLO11s).